## Cell 1 — Cài đặt thư viện

In [ ]:
!pip install -q streamlit pyngrok transformers torch torchvision Pillow nltk requests
print('Xong!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 73.1 MB/s eta 0:00:00
Xong!


## Cell 2 — Mount Drive & Giải nén Model

In [ ]:
from google.colab import drive
import os, zipfile, shutil
from pathlib import Path

# Mount Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive mounted!')

# ── Sửa đường dẫn nếu cần ────────────────────────────────
BLIP_ZIP   = '/content/drive/MyDrive/blip_best_model.zip'
CNN_PT     = '/content/drive/MyDrive/cnn_lstm_best_model.pt'
VOCAB_JSON = '/content/drive/MyDrive/vocab.json'
BLIP_OUT   = '/content/models/blip_best_model'
# ─────────────────────────────────────────────────────────

# Giải nén BLIP
if not Path(BLIP_OUT).exists():
    print('Giải nén BLIP')
    Path(BLIP_OUT).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(BLIP_ZIP, 'r') as z:
        z.extractall(BLIP_OUT)
    print(' Giải nén xong!')
else:
    print('BLIP đã giải nén sẵn!')

# Kiểm tra file
files = list(Path(BLIP_OUT).rglob('*'))
print(f'   {len(files)} files trong BLIP folder:')
for f in files:
    print(f'     {f.name}  ({f.stat().st_size/1e6:.1f} MB)')

# Copy CNN+LSTM
if Path(CNN_PT).exists():
    shutil.copy(CNN_PT, '/content/cnn_lstm_best_model.pt')
    print(' CNN+LSTM copied')
else:
    print(f' Không thấy {CNN_PT}')

if Path(VOCAB_JSON).exists():
    shutil.copy(VOCAB_JSON, '/content/vocab.json')
    print('vocab.json copied')
else:
    print(f'Không thấy {VOCAB_JSON}')

# Set env
os.environ['BLIP_MODEL_PATH'] = BLIP_OUT
print(f'\nBLIP_MODEL_PATH = {BLIP_OUT}')
print('Sẵn sàng chạy Streamlit!')

Mounted at /content/drive
Giải nén BLIP
 Giải nén xong!
   6 files trong BLIP folder:
     model.safetensors  (989.8 MB)
     tokenizer.json  (0.7 MB)
     generation_config.json  (0.0 MB)
     config.json  (0.0 MB)
     processor_config.json  (0.0 MB)
     tokenizer_config.json  (0.0 MB)
 CNN+LSTM copied
vocab.json copied

BLIP_MODEL_PATH = /content/models/blip_best_model
Sẵn sàng chạy Streamlit!


## Cell 3 — Tạo app.py

In [ ]:
# Ghi app.py ra file — dùng double quotes để tránh conflict
app_lines = [
    'import os, io, json, requests\n',
    'from pathlib import Path\n',
    'from PIL import Image\n',
    'import torch\n',
    'import torch.nn as nn\n',
    'import torchvision.transforms as transforms\n',
    'import torchvision.models as models\n',
    'import nltk\n',
    'from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction\n',
    'nltk.download("punkt", quiet=True)\n',
    'nltk.download("punkt_tab", quiet=True)\n',
    'import streamlit as st\n',
    'from transformers import BlipProcessor, BlipForConditionalGeneration\n',
    '\n',
    'st.set_page_config(page_title="Image Captioning", page_icon="🖼️", layout="wide")\n',
    '\n',
    'st.markdown("""\n',
    '<style>\n',
    'html, body, [class*="css"] { font-family: Inter, sans-serif; }\n',
    '.main-title {\n',
    '    font-size: 2.2rem; font-weight: 700;\n',
    '    background: linear-gradient(135deg, #2563eb, #f97316);\n',
    '    -webkit-background-clip: text; -webkit-text-fill-color: transparent;\n',
    '}\n',
    '.blip-card {\n',
    '    background: linear-gradient(135deg,#eff6ff,#dbeafe);\n',
    '    border: 2px solid #3b82f6; border-radius: 14px;\n',
    '    padding: 1.2rem 1.4rem; margin-bottom: 1rem;\n',
    '}\n',
    '.cnn-card {\n',
    '    background: linear-gradient(135deg,#fff7ed,#fed7aa);\n',
    '    border: 2px solid #f97316; border-radius: 14px;\n',
    '    padding: 1.2rem 1.4rem; margin-bottom: 1rem;\n',
    '}\n',
    '.model-label { font-size:0.85rem; font-weight:700; letter-spacing:0.08em;\n',
    '    text-transform:uppercase; margin-bottom:0.4rem; }\n',
    '.caption-text { font-size:1.05rem; line-height:1.6; color:#1f2937; }\n',
    '.bleu-box { background:#f0fdf4; border:1.5px solid #22c55e;\n',
    '    border-radius:10px; padding:1rem 1.2rem; margin-top:1rem; }\n',
    '.stButton > button { background: linear-gradient(135deg,#2563eb,#1d4ed8);\n',
    '    color:white; border:none; border-radius:8px;\n',
    '    padding:0.6rem 2rem; font-weight:600; width:100%; }\n',
    '</style>\n',
    '""", unsafe_allow_html=True)\n',
    '\n',
    'DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n',
    '\n',
    'class Vocabulary:\n',
    '    PAD,SOS,EOS,UNK = 0,1,2,3\n',
    '    def __init__(self):\n',
    '        self.word2idx={"<PAD>":0,"<SOS>":1,"<EOS>":2,"<UNK>":3}\n',
    '        self.idx2word={v:k for k,v in self.word2idx.items()}\n',
    '    def load(self,path):\n',
    '        with open(path) as f: self.word2idx=json.load(f)\n',
    '        self.idx2word={int(v):k for k,v in self.word2idx.items()}\n',
    '    def decode(self,ids):\n',
    '        w=[]\n',
    '        for i in ids:\n',
    '            if i==self.EOS: break\n',
    '            if i not in (self.PAD,self.SOS): w.append(self.idx2word.get(i,"<UNK>"))\n',
    '        return " ".join(w)\n',
    '    def __len__(self): return len(self.word2idx)\n',
    '\n',
    'class CNNEncoder(nn.Module):\n',
    '    def __init__(self,embed_dim=256):\n',
    '        super().__init__()\n',
    '        r=models.resnet50(weights=None)\n',
    '        self.backbone=nn.Sequential(*list(r.children())[:-1])\n',
    '        self.fc=nn.Linear(2048,embed_dim)\n',
    '        self.bn=nn.BatchNorm1d(embed_dim)\n',
    '    def forward(self,x):\n',
    '        with torch.no_grad(): f=self.backbone(x)\n',
    '        return self.bn(self.fc(f.view(f.size(0),-1)))\n',
    '\n',
    'class LSTMDecoder(nn.Module):\n',
    '    def __init__(self,e,h,v):\n',
    '        super().__init__()\n',
    '        self.embed=nn.Embedding(v,e,padding_idx=0)\n',
    '        self.lstm=nn.LSTM(e,h,1,batch_first=True)\n',
    '        self.fc_out=nn.Linear(h,v)\n',
    '        self.init_h=nn.Linear(e,h)\n',
    '        self.init_c=nn.Linear(e,h)\n',
    '    @torch.no_grad()\n',
    '    def generate(self,enc,vocab,max_len=40):\n',
    '        B=enc.size(0)\n',
    '        h=self.init_h(enc).unsqueeze(0); c=self.init_c(enc).unsqueeze(0)\n',
    '        tok=torch.full((B,1),vocab.SOS,dtype=torch.long,device=enc.device)\n',
    '        res=[[] for _ in range(B)]; done=[False]*B\n',
    '        for _ in range(max_len):\n',
    '            out,(h,c)=self.lstm(self.embed(tok),(h,c))\n',
    '            tok=self.fc_out(out.squeeze(1)).argmax(dim=1,keepdim=True)\n',
    '            for i in range(B):\n',
    '                if not done[i]:\n',
    '                    t=tok[i].item()\n',
    '                    if t==vocab.EOS: done[i]=True\n',
    '                    else: res[i].append(t)\n',
    '            if all(done): break\n',
    '        return [vocab.decode(r) for r in res]\n',
    '\n',
    'class CNNLSTMCaptioner(nn.Module):\n',
    '    def __init__(self,e,h,v):\n',
    '        super().__init__()\n',
    '        self.encoder=CNNEncoder(e); self.decoder=LSTMDecoder(e,h,v)\n',
    '    def forward(self,imgs,vocab): return self.decoder.generate(self.encoder(imgs),vocab)\n',
    '\n',
    'cnn_tf=transforms.Compose([\n',
    '    transforms.Resize((224,224)),transforms.ToTensor(),\n',
    '    transforms.Normalize([.485,.456,.406],[.229,.224,.225])\n',
    '])\n',
    '\n',
    '@st.cache_resource(show_spinner="⏳ Loading BLIP từ local (~30s)...")\n',
    'def load_blip():\n',
    '    mid=os.environ.get("BLIP_MODEL_PATH","Salesforce/blip-image-captioning-base")\n',
    '    proc=BlipProcessor.from_pretrained(mid)\n',
    '    dtype=torch.float16 if DEVICE.type=="cuda" else torch.float32\n',
    '    mdl=BlipForConditionalGeneration.from_pretrained(mid,torch_dtype=dtype).to(DEVICE)\n',
    '    mdl.eval()\n',
    '    dummy=torch.zeros(1,3,384,384,dtype=dtype).to(DEVICE)\n',
    '    with torch.no_grad(): mdl.generate(pixel_values=dummy,max_new_tokens=5)\n',
    '    return proc,mdl\n',
    '\n',
    '@st.cache_resource(show_spinner="⏳ Loading CNN+LSTM...")\n',
    'def load_cnn():\n',
    '    vp="/content/vocab.json"; mp="/content/cnn_lstm_best_model.pt"\n',
    '    if not (Path(vp).exists() and Path(mp).exists()): return None,None\n',
    '    v=Vocabulary(); v.load(vp)\n',
    '    m=CNNLSTMCaptioner(256,512,len(v)).to(DEVICE)\n',
    '    ckpt=torch.load(mp,map_location=DEVICE)\n',
    '    m.load_state_dict(ckpt["model_state_dict"]); m.eval()\n',
    '    return v,m\n',
    '\n',
    'def blip_cap(img,proc,mdl):\n',
    '    inp=proc(images=img,return_tensors="pt").to(DEVICE)\n',
    '    dtype=torch.float16 if DEVICE.type=="cuda" else torch.float32\n',
    '    inp={k:v.to(dtype) if v.dtype==torch.float32 else v for k,v in inp.items()}\n',
    '    with torch.no_grad():\n',
    '        out=mdl.generate(**inp,max_new_tokens=50,num_beams=4,no_repeat_ngram_size=3)\n',
    '    return proc.decode(out[0],skip_special_tokens=True)\n',
    '\n',
    'def cnn_cap(img,vocab,mdl):\n',
    '    if mdl is None: return "⚠️ Thiếu file model"\n',
    '    with torch.no_grad(): return mdl(cnn_tf(img).unsqueeze(0).to(DEVICE),vocab)[0]\n',
    '\n',
    'def bleu(ref,hyp):\n',
    '    sf=SmoothingFunction().method1\n',
    '    return sentence_bleu([nltk.word_tokenize(ref.lower())],\n',
    '                          nltk.word_tokenize(hyp.lower()),smoothing_function=sf)\n',
    '\n',
    'def show_results(img,ref=""):\n',
    '    c1,c2=st.columns(2,gap="large")\n',
    '    with c1: st.image(img,use_container_width=True)\n',
    '    with c2:\n',
    '        with st.spinner("Đang sinh caption..."):\n',
    '            bc=blip_cap(img,proc,blip_mdl)\n',
    '            cc=cnn_cap(img,vocab,cnn_mdl)\n',
    '        st.markdown(f"<div class=blip-card><div class=model-label style=color:#2563eb>🔵 BLIP</div><div class=caption-text>{bc}</div></div>",unsafe_allow_html=True)\n',
    '        st.markdown(f"<div class=cnn-card><div class=model-label style=color:#f97316>🟠 CNN+LSTM</div><div class=caption-text>{cc}</div></div>",unsafe_allow_html=True)\n',
    '        if ref.strip():\n',
    '            bb=bleu(ref,bc); cb=bleu(ref,cc)\n',
    '            w="🔵 BLIP" if bb>=cb else "🟠 CNN+LSTM"\n',
    '            st.markdown(f"<div class=bleu-box>📊 BLEU | 🏆 {w} tốt hơn<br>🔵 {bb*100:.1f}% | 🟠 {cb*100:.1f}%</div>",unsafe_allow_html=True)\n',
    '\n',
    'st.markdown("<div class=main-title>🖼️ Image Captioning</div>",unsafe_allow_html=True)\n',
    'st.markdown("**BLIP vs CNN+LSTM** — Fine-tuned trên Flickr30k")\n',
    '\n',
    'with st.spinner("⏳ Khởi động models..."):\n',
    '    proc,blip_mdl=load_blip()\n',
    '    vocab,cnn_mdl=load_cnn()\n',
    'st.success(f"✅ Sẵn sàng! Device: {str(DEVICE).upper()}")\n',
    '\n',
    'tab1,tab2=st.tabs(["📤 Upload Ảnh","🌐 Từ URL"])\n',
    '\n',
    'with tab1:\n',
    '    up=st.file_uploader("Chọn ảnh",type=["jpg","jpeg","png","webp"])\n',
    '    ref1=st.text_input("Reference caption (tuỳ chọn)",placeholder="Nhập để tính BLEU...",key="r1")\n',
    '    if st.button("🚀 Sinh Caption",key="b1"):\n',
    '        if up: show_results(Image.open(up).convert("RGB"),ref1)\n',
    '        else: st.warning("⬆️ Hãy upload ảnh trước!")\n',
    '\n',
    'with tab2:\n',
    '    url=st.text_input("URL ảnh",placeholder="https://...")\n',
    '    ref2=st.text_input("Reference caption (tuỳ chọn)",placeholder="Nhập để tính BLEU...",key="r2")\n',
    '    if st.button(" Sinh Caption",key="b2"):\n',
    '        if url.strip():\n',
    '            try:\n',
    '                r=requests.get(url.strip(),timeout=10,headers={"User-Agent":"Mozilla/5.0"})\n',
    '                show_results(Image.open(io.BytesIO(r.content)).convert("RGB"),ref2)\n',
    '            except Exception as e: st.error(f"❌ {e}")\n',
    '        else: st.warning("🌐 Hãy nhập URL trước!")\n',
]

with open('/content/app.py', 'w') as f:
    f.writelines(app_lines)
print('✅ app.py created!')

✅ app.py created!


## Cell 4 — Chạy Streamlit + Lấy link public

> Lấy ngrok token miễn phí: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
import subprocess, time, os
from pyngrok import ngrok

# ── ĐIỀN TOKEN VÀO ĐÂY ───────────────────────────────────
NGROK_TOKEN = '3DnjI1tDrM0ELsYg6wmyW8N4vJO_72ggvb33SRmYhfBVzEm1c'   # ← paste token từ dashboard.ngrok.com
# ─────────────────────────────────────────────────────────

if not NGROK_TOKEN:
    print('Chưa có ngrok token!')
    print('→ Vào https://dashboard.ngrok.com/get-started/your-authtoken')
    print('→ Đăng ký miễn phí → copy token → paste vào NGROK_TOKEN')
else:
    # Kiểm tra model đã load chưa
    blip_path = os.environ.get('BLIP_MODEL_PATH', '')
    if not blip_path:
        print('BLIP_MODEL_PATH chưa set — hãy chạy lại Cell 2 trước!')
    else:
        print(f'BLIP sẽ load từ: {blip_path}')

        # Kill session cũ
        os.system('pkill -f streamlit 2>/dev/null')
        try: ngrok.kill()
        except: pass
        time.sleep(2)

        # Chạy Streamlit
        proc = subprocess.Popen(
            ['streamlit', 'run', '/content/app.py',
             '--server.port', '8501',
             '--server.headless', 'true',
             '--server.enableCORS', 'false',
             '--server.enableXsrfProtection', 'false'],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        time.sleep(5)

        # Tạo ngrok tunnel
        ngrok.set_auth_token(NGROK_TOKEN)
        tunnel = ngrok.connect(8501)

        print('\n' + '='*55)
        print('App đang chạy!')
        print(f'Link: {tunnel.public_url}')
        print('='*55)

BLIP sẽ load từ: /content/models/blip_best_model

App đang chạy!
Link: https://evaluator-exciting-immersion.ngrok-free.dev
